# Local Training Script Testing

This notebook tests the training pipeline locally before submitting expensive GPU jobs to Azure ML.

## What This Notebook Does

1. **Validates Configuration**: Ensures training config has all required parameters
2. **Tests Data Loading**: Verifies training data can be loaded and tokenized correctly
3. **Initializes Model**: Loads base model and applies LoRA adapters for fine-tuning
4. **Runs Mini Training**: Executes a few training steps locally to catch bugs early
5. **Validates Checkpointing**: Ensures model can be saved and loaded correctly

## Why Test Locally First?

**Cost Savings:**
- Azure ML GPU clusters cost $3-12/hour
- Local testing catches configuration errors in minutes instead of hours
- Prevents wasted compute quota on broken training jobs

**Faster Debugging:**
- Immediate feedback on errors
- Can use IDE debugger for complex issues
- Iterate quickly on hyperparameters

**Validation:**
- Confirms data format is correct
- Verifies tokenization doesn't truncate important text
- Ensures LoRA configuration is compatible with model architecture

## What Gets Tested

### Data Pipeline
- JSONL file can be parsed
- Prompt/completion fields exist and are non-empty
- Tokenization produces valid input tensors
- Batch creation works without errors

### Model Configuration
- Base model loads from Hugging Face or local path
- LoRA adapters apply correctly to target modules
- Forward pass completes without shape mismatches
- Loss computation works

### Training Loop
- Optimizer initializes with correct learning rate
- Gradient computation doesn't produce NaNs
- Checkpoint saving/loading preserves model state

## Limitations of Local Testing

⚠️ **This notebook runs on CPU** - training will be slow (minutes per step)
⚠️ **Limited memory** - only tests with small batch sizes and short sequences
⚠️ **Not a full training run** - just validates the pipeline works

For actual model training, use notebook `06-submit-training-job.ipynb` which runs on GPU clusters.

## Prerequisites

- Completed notebook `02-prepare-data.ipynb`
- Completed notebook `04-download-model.ipynb`
- Training data available at path specified in `configs/training_config.yaml`
- Sufficient local disk space (~5GB) for model weights

## Expected Duration

~5-10 minutes for data loading and mini training

## 1. Setup Environment

**Why connect to Azure ML?** Even though we're testing locally, we need Azure ML client to access workspace configs and verify our credentials are valid for the actual training job submission later.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import yaml

# Initialize Azure ML client
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print(f"Connected to workspace: {ml_client.workspace_name}")
print(f"Project root: {project_root}")

## 2. Load Training Configuration

**Why load config?** The training config centralizes all hyperparameters (learning rate, batch size, epochs, LoRA settings). Using YAML keeps configs version-controlled and easy to experiment with.

In [ ]:
# Load config
config_path = project_root / "configs" / "training_config.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

print("Training Configuration:")
print(f"  Model: {config['model']['name_or_path']}")
print(f"  Epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['per_device_train_batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  LoRA rank: {config['lora']['r']}")
print(f"  LoRA alpha: {config['lora']['alpha']}")

## 3. Test Data Loading

**Why test data loading?** This is the most common source of training failures. We verify the JSONL format is correct, tokenization works, and batches can be created before spending money on GPU time.

In [ ]:
from src.data.dataset_loader import ConversationDataset, load_tokenizer

# Load tokenizer
model_path = config['model']['name_or_path']
tokenizer = load_tokenizer(model_path)

print(f"Tokenizer loaded:")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Pad token: {tokenizer.pad_token}")

# Load small sample of training data
train_file = project_root / config['data']['train_file']

if train_file.exists():
    dataset = ConversationDataset(
        data_path=train_file,
        tokenizer=tokenizer,
        max_length=config['data']['max_seq_length'],
    )

    print(f"\nDataset loaded: {len(dataset)} examples")

    # Show first example
    example = dataset[0]
    print(f"\nExample shape:")
    print(f"  input_ids: {example['input_ids'].shape}")
    print(f"  attention_mask: {example['attention_mask'].shape}")
    print(f"  labels: {example['labels'].shape}")
else:
    print(f"⚠️  Training file not found: {train_file}")
    print("Please run notebook 02 to prepare training data.")

## 4. Verify Model Setup

In [ ]:
from src.training.trainer import TrainingConfig, setup_lora_model

# Create training config
training_config = TrainingConfig(
    model_name_or_path=model_path,
    use_lora=config['lora']['enabled'],
    lora_r=config['lora']['r'],
    lora_alpha=config['lora']['alpha'],
    lora_dropout=config['lora']['dropout'],
    num_epochs=1,  # Short test
    batch_size=1,
)

print("Training configuration created")
print("\nNote: Full model setup requires GPU and ~30GB memory")
print("To test model loading, run on Azure ML compute cluster.")

## 5. Run Local Training Test (Optional)

⚠️ **Warning**: This requires a GPU with at least 24GB memory.

Skip this section if running on a machine without GPU.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Uncomment to run local training test
    # !python src/training/train.py --config configs/training_config.yaml
else:
    print("No GPU available. Please use Azure ML compute for training.")

## 6. Submit Training Job to Azure ML

Submit the training job to Azure ML compute cluster.

In [ ]:
from azure.ai.ml import command
from azure.ai.ml.entities import Environment

# Define compute target
compute_name = config.get('azure_ml', {}).get('compute_target', 'gpu-cluster')

# Create training job
job = command(
    code=str(project_root),
    command="python src/training/train.py --config configs/training_config.yaml",
    environment="AzureML-pytorch-2.1-cuda11.8:1",  # Pre-built PyTorch environment
    compute=compute_name,
    display_name="phi-4-finetuning",
    experiment_name="phi-4-training",
)

print(f"Submitting training job to: {compute_name}")
print("\nNote: Uncomment below to actually submit the job")

# Uncomment to submit
# returned_job = ml_client.jobs.create_or_update(job)
# print(f"Job submitted: {returned_job.name}")
# print(f"Studio URL: {returned_job.studio_url}")

## 7. Monitor Training Progress

In [ ]:
# Monitor specific job
# job_name = "<job-name>"  # Replace with actual job name
# job = ml_client.jobs.get(job_name)
# print(f"Job status: {job.status}")
# print(f"Studio URL: {job.studio_url}")

# List recent training jobs
jobs = ml_client.jobs.list(max_results=5)
print("Recent training jobs:")
for job in jobs:
    print(f"  {job.name}: {job.status}")

## Next Steps

After training completes:
1. **Evaluate Model**: Run notebook 06 to evaluate model performance
2. **Review Metrics**: Check MLflow tracking for training metrics
3. **Download Model**: Download trained model from Azure ML

## Troubleshooting

### Out of Memory (OOM)
- Reduce `per_device_train_batch_size` to 1 or 2
- Increase `gradient_accumulation_steps`
- Enable 4-bit quantization: `use_4bit: true`
- Enable gradient checkpointing: `gradient_checkpointing: true`

### Slow Training
- Ensure using GPU compute
- Check `dataloader_num_workers` (try 4-8)
- Enable mixed precision: `bf16: true`
- Use Flash Attention 2 if available

### Data Loading Errors
- Verify training data exists in `data/processed/`
- Check JSONL format with `jq` command
- Review data preparation notebook (02)

### Checkpoint Issues
- Check `save_steps` value (should be reasonable)
- Verify output directory permissions
- Check disk space for checkpoints

For more help, see the training documentation in `docs/training.md`.